In [7]:
!pip install -q marimo
import marimo as mo

In [1]:
!pip install -q git+https://github.com/fabienfrfr/Kairos@dev

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 87.0 MB/s eta 0:00:00:00:010:01


# 🌀 Kairos — Training Notebook
Build · Inspect · Train · Checkpoint


⚠️ **Work in Progress (WIP)**

This notebook is actively under development. Some components (training loop, logging, checkpointing, etc.) may be incomplete, experimental, or subject to change. Use for testing and exploration only.

In [2]:
import os
import math
import time
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import torch
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader

from transformers import TrainingArguments

from kairos.modeling import KairosConfig, KairosDiffusionLLM, ConvCodec
from kairos.attentions import KairosCache
from kairos.tokenizer import KairosTokenizer
from kairos.dataset import KairosPretrainingDataset
from kairos.trainer import KairosDiffusionTrainer

2026-07-01 20:13:02.020462: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782936782.220692      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782936782.275222      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782936782.728379      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782936782.728428      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782936782.728431      58 computation_placer.cc:177] computation placer alr

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


*Estimate pretraining compute time*

Based on Training Compute-Optimal Large Language Models : https://arxiv.org/abs/2203.15556

Verify calculus for token/s

In [4]:
GPU_FLOPS = {
    "T4": 65,
    "RTX_5060Ti": 200,
    "RTX_5070Ti": 300,
    "A100": 312,
    "H100": 1000,
}

GPU_BW = {
    "T4": 300e9,
    "RTX_5060Ti": 600e9,
    "RTX_5070Ti": 800e9,
    "A100": 1500e9,
    "H100": 3000e9,
}


gpu = "T4" # "GPU"
eff = 0.3 # "GPU efficiency"
params = 2 # "Active params (millions)"
tokens = 1 # "Training tokens (billions)" --> keep-it-simple

In [5]:
flops = GPU_FLOPS[gpu] * 1e12
bw = GPU_BW[gpu]

params_total = params * 1e6
tokens_total = tokens * 1e9

n_ops = 6  # compute cost (qkvo+ffn+1)

# --- compute bound ---
tok_s_compute = (flops / (n_ops * params_total)) * eff

# --- memory bound ---
bytes_per_token = 4 * params_total  # fp16 rule-of-thumb
tok_s_memory = bw / bytes_per_token

# --- real ---
tok_s_real = min(tok_s_compute, tok_s_memory)
time_days = tokens_total / tok_s_real / 86400

print(f"""
🧠 Compute-bound` {tok_s_compute:.0f} tok/s`
💾 Memory-bound` {tok_s_memory:.0f} tok/s`
⚖️ Real throughput` {tok_s_real:.0f} tok/s`
⏱️ Training time` {time_days:.1f} days`
""")


🧠 Compute-bound` 1625000 tok/s`
💾 Memory-bound` 37500 tok/s`
⚖️ Real throughput` 37500 tok/s`
⏱️ Training time` 0.3 days`



## ⚙️ Model Configuration

In [6]:
cfg_d_model = 256  # "d_model (hidden size)"
cfg_n_heads = 4    # "n_heads"
cfg_n_layers = 4   # "n_layers"
cfg_window = 64    # "SWA window size"
cfg_stride = 3     # "ConvCodec stride"
cfg_experts = 3    # "MoE experts (0 = dense FFN)"

In [7]:
config = KairosConfig(
    d_model=cfg_d_model,
    n_heads=cfg_n_heads,
    n_layers=cfg_n_layers,
    window_size=cfg_window,
    stride=cfg_stride,
    num_experts=cfg_experts if cfg_experts > 0 else 8,
    num_experts_per_tok=2,
)

num_experts_arg = cfg_experts if cfg_experts > 0 else None

model = KairosDiffusionLLM(
    config,
    vocab_size=259,
    num_experts=num_experts_arg,
).to(device)

total_params   = sum(p.numel() for p in model.parameters())
active_params  = sum(
    p.numel() for n, p in model.named_parameters()
    if "experts" not in n or any(f"experts.{i}" in n for i in range(config.num_experts_per_tok))
)

print(
f"Total params: `{total_params/1e6:.2f}M`  \n"
f"Active params (est.): `{active_params/1e6:.2f}M`  \n"
f"Layers config: `{config.layers_config}`  \n"
f"MoE: `{'Yes — ' + str(cfg_experts) + ' experts' if num_experts_arg else 'No — dense FFN'}`"
)

[transformers] The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Total params: `60.40M`  
Active params (est.): `3.78M`  
Layers config: `['ld', 'ld', 'ld', 'ld']`  
MoE: `Yes — 3 experts`


## 🔬 Architecture Inspection

In [8]:
rows = []
for name, param in model.named_parameters():
    rows.append({
        "Layer": name,
        "Shape": str(list(param.shape)),
        "Params": f"{param.numel():,}",
        "Trainable": "✅" if param.requires_grad else "❌",
        "dtype": str(param.dtype).replace("torch.", ""),
    })

df = pd.DataFrame(rows)
display(df)

,Layer,Shape,Params,Trainable,dtype
0,codec.enc.weight,"[256, 1, 5]","1,280",✅,float32
1,codec.enc.bias,[256],256,✅,float32
2,codec.dec.weight,"[256, 1, 5]","1,280",✅,float32
3,codec.dec.bias,[256],256,✅,float32
4,token_embed.embed.weight,"[259, 256]","66,304",✅,float32
...,...,...,...,...,...
88,backbone.layers.3.ffn.shared_experts.down_proj...,"[256, 2048]","524,288",✅,float32
89,backbone.norm.weight,[256],256,✅,float32
90,backbone.aggregator.w,[256],256,✅,float32
91,backbone.aggregator.key_norm.weight,[256],256,✅,float32


In [9]:
# Dry-run forward to check shapes
_x = torch.randint(0, 259, (1, 12)).to(device)
with torch.no_grad():
    _out = model(input_ids=_x)

print(
f"**Dry-run input:** `(1, 12)` tokens  \n"
f"**After codec encode:** `(1, {12 // config.stride}, {config.hidden_size})`  \n"
f"**Output logits:** `{list(_out.logits.shape)}`  \n"
f"**No NaN:** `{not torch.isnan(_out.logits).any().item()}`"
)

**Dry-run input:** `(1, 12)` tokens  
**After codec encode:** `(1, 4, 256)`  
**Output logits:** `[1, 12, 259]`  
**No NaN:** `False`


## 🏋️ Training Configuration

In [18]:
train_lr          = 3e-4 # "Learning rate"
train_batch       = 8  # "Batch size"
train_epochs      = 5   # "Epochs"
train_max_len     = 128 # "Max sequence length"
train_save_steps  = 50  # "Save every N steps"
train_log_steps   = 10  # "Log every N steps"
train_output_dir  = "checkpoints/kairos" # "Output directory"
train_run_name    = "run_01" # "Run name (TensorBoard)"

## 📦 Dataset

In [12]:
dataset_source = "ffurfaro/keep-it-simple"
dataset_source

'ffurfaro/keep-it-simple'

In [13]:
custom_texts_input = "Paris is the capital of France.\nThe Earth orbits the Sun.\nWater boils at 100 degrees Celsius."
if custom_texts_input:
    custom_texts_input

In [ ]:
tokenizer = KairosTokenizer()

texts = None
if dataset_source == "custom texts" and custom_texts_input:
    texts = [t for t in custom_texts_input.split("\n") if t.strip()]


dataset = KairosPretrainingDataset(
    texts=texts,
    tokenizer=tokenizer,
    max_len=train_max_len,
)

print(
f"**Samples:** `{len(dataset)}`  |  **Max len:** `{train_max_len}`  |  **Source:** `{dataset_source}`"
)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

data/auto_math_text/train-00000-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

data/auto_math_text/train-00011-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/auto_math_text/train-00012-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/auto_math_text/train-00013-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/auto_math_text/train-00014-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/auto_math_text/train-00015-of-00018(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

## 🚀 Train

In [ ]:
run_button = mo.ui.run_button(label="▶ Start Training")
run_button

In [ ]:
if not run_button.value:
    mo.stop(True, mo.callout(mo.md("Click **▶ Start Training** to begin."), kind="neutral"))

# ── dirs ──
run_dir = Path(train_output_dir.value) / train_run_name.value
tb_dir  = run_dir / "tensorboard"
ckpt_dir = run_dir / "checkpoints"
run_dir.mkdir(parents=True, exist_ok=True)
tb_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)

# ── save config ──
import json as _json
with open(run_dir / "config.json", "w") as _f:
    _json.dump({
        "d_model": config.hidden_size,
        "n_heads": config.num_attention_heads,
        "n_layers": config.num_hidden_layers,
        "sliding_window_size": config.sliding_window_size,
        "stride": config.stride,
        "vocab_size": 259,
    }, _f, indent=2)

writer = SummaryWriter(log_dir=str(tb_dir))

training_args = TrainingArguments(
    output_dir=str(ckpt_dir),
    num_train_epochs=train_epochs.value,
    per_device_train_batch_size=train_batch.value,
    learning_rate=train_lr.value,
    logging_steps=train_log_steps.value,
    save_steps=train_save_steps.value,
    save_total_limit=3,
    report_to=[],          # on gère TensorBoard manuellement
    dataloader_pin_memory=(device == "cuda"),
    remove_unused_columns=False,
    no_cuda=(device != "cuda"),
)

trainer = KairosDiffusionTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

# ── custom loop with marimo progress + TensorBoard ──
loader = DataLoader(
    dataset,
    batch_size=train_batch.value,
    shuffle=True,
    pin_memory=(device == "cuda"),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=train_lr.value)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=train_epochs.value * len(loader),
    eta_min=train_lr.value * 0.1,
)

global_step = 0
best_loss   = float("inf")
log_rows    = []

model.train()

with mo.status.progress_bar(
    total=train_epochs.value * len(loader),
    title="Training Kairos",
    subtitle="epoch 1",
) as _bar:

    for epoch in range(1, train_epochs.value + 1):
        epoch_loss = 0.0

        for step, batch in enumerate(loader, 1):
            # move to device
            batch = {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}

            optimizer.zero_grad()

            loss = trainer.compute_loss(model, batch)
            loss.backward()

            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            loss_val   = loss.item()
            epoch_loss += loss_val
            global_step += 1

            # ── TensorBoard ──
            if global_step % train_log_steps.value == 0:
                lr_now = scheduler.get_last_lr()[0]
                writer.add_scalar("train/loss",    loss_val,   global_step)
                writer.add_scalar("train/lr",      lr_now,     global_step)
                writer.add_scalar("train/epoch",   epoch,      global_step)

                # gradient norm
                grad_norm = math.sqrt(sum(
                    p.grad.norm().item() ** 2
                    for p in model.parameters()
                    if p.grad is not None
                ))
                writer.add_scalar("train/grad_norm", grad_norm, global_step)

                log_rows.append({
                    "step":  global_step,
                    "epoch": epoch,
                    "loss":  f"{loss_val:.4f}",
                    "lr":    f"{lr_now:.2e}",
                    "grad_norm": f"{grad_norm:.3f}",
                })

            # ── Checkpoint ──
            if global_step % train_save_steps.value == 0:
                ckpt_path = ckpt_dir / f"step_{global_step:06d}.pt"
                torch.save({
                    "step":            global_step,
                    "epoch":           epoch,
                    "model_state":     model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "loss":            loss_val,
                    "config":          config.to_dict(),
                }, ckpt_path)
                writer.add_text(
                    "checkpoints",
                    f"Saved `{ckpt_path.name}` — loss {loss_val:.4f}",
                    global_step,
                )

            _bar.update(subtitle=f"epoch {epoch}/{train_epochs.value} | loss {loss_val:.4f}")

        avg_loss = epoch_loss / len(loader)
        writer.add_scalar("train/epoch_avg_loss", avg_loss, epoch)

        # ── Best checkpoint ──
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_path = ckpt_dir / "best.pt"
            torch.save({
                "step":        global_step,
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "loss":        best_loss,
                "config":      config.to_dict(),
            }, best_path)
            writer.add_text("checkpoints", f"🏆 New best at epoch {epoch} — loss {best_loss:.4f}", global_step)

writer.flush()
writer.close()

mo.callout(
    mo.md(
        f"✅ **Training complete**  \n"
        f"Steps: `{global_step}` | Best loss: `{best_loss:.4f}`  \n"
        f"Checkpoints: `{ckpt_dir}`  \n"
        f"TensorBoard: `tensorboard --logdir {tb_dir}`"
    ),
    kind="success",
)

In [ ]:
if not run_button.value or not log_rows:
    mo.stop(True)

mo.vstack([
    mo.md("## 📊 Training Logs"),
    mo.ui.table(log_rows, label="Step logs", pagination=True, page_size=20),
])

## 💾 Checkpoint Browser

In [ ]:
ckpt_browser_dir = mo.ui.text(
    value="checkpoints/kairos/run_01/checkpoints",
    label="Checkpoint directory",
)
ckpt_browser_dir

In [ ]:
_ckpt_path = Path(ckpt_browser_dir.value)

if not _ckpt_path.exists():
    mo.callout(mo.md(f"Directory `{_ckpt_path}` not found."), kind="warn")
else:
    _files = sorted(_ckpt_path.glob("*.pt"))
    if not _files:
        mo.callout(mo.md("No checkpoints found."), kind="warn")
    else:
        _rows = []
        for _f in _files:
            try:
                _ck = torch.load(_f, map_location="cpu", weights_only=True)
                _rows.append({
                    "File":    _f.name,
                    "Step":    _ck.get("step", "?"),
                    "Epoch":   _ck.get("epoch", "?"),
                    "Loss":    f"{_ck.get('loss', 0):.4f}",
                    "Size":    f"{_f.stat().st_size / 1e6:.1f} MB",
                })
            except Exception as _e:
                _rows.append({"File": _f.name, "Step": "error", "Epoch": "?", "Loss": "?", "Size": "?"})

        mo.ui.table(_rows, label="Available checkpoints")

## ♻️ Resume from Checkpoint

In [ ]:
resume_path   = mo.ui.text(value="", label="Checkpoint path (.pt)")
resume_button = mo.ui.run_button(label="Load checkpoint")
mo.hstack([resume_path, resume_button])

In [ ]:
if not resume_button.value or not resume_path.value:
    mo.stop(True)

_path = Path(resume_path.value)
if not _path.exists():
    mo.callout(mo.md(f"File not found: `{_path}`"), kind="danger")
else:
    _ck = torch.load(_path, map_location="cpu")
    model.load_state_dict(_ck["model_state"])
    if "optimizer_state" in _ck:
        optimizer.load_state_dict(_ck["optimizer_state"])
    if "scheduler_state" in _ck:
        scheduler.load_state_dict(_ck["scheduler_state"])

    mo.callout(
        mo.md(
            f"✅ Loaded `{_path.name}`  \n"
            f"Step: `{_ck.get('step', '?')}` | Epoch: `{_ck.get('epoch', '?')}` | Loss: `{_ck.get('loss', 0):.4f}`"
        ),
        kind="success",
    )

In [ ]:
if not run_button.value:
    mo.stop(True)

mo.callout(
    mo.md(
        f"## 📈 TensorBoard\n\n"
        f"Lance dans un terminal :\n\n"
        f"```bash\ntensorboard --logdir {tb_dir} --port 6006\n```\n\n"
        f"Puis ouvre [http://localhost:6006](http://localhost:6006)"
    ),
    kind="info",
)